<a href="https://colab.research.google.com/github/itinasharma/MachineLearning/blob/main/faiss_vector_search_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
!pip install faiss-cpu

**Phase 0 — Setup + Metadata**
What we are doing:

We install FAISS and create a place to store product data.

**Key idea:**

FAISS only gives IDs, not actual data.

In [42]:
import faiss
import numpy as np
import os

metadata_store = {
    0: {"title": "Wireless Noise-Cancelling Headphones", "price": 299.99, "category": "audio"},
    1: {"title": "Bluetooth Speaker Portable", "price": 79.99, "category": "audio"},
    2: {"title": "Mechanical Keyboard TKL", "price": 149.00, "category": "peripherals"},
}

Phase 1 — Building the Index
Why do we need Phase 1?

Because searching every vector directly is too slow.

Imagine:

You have 100,000 products
Each product = 1536 numbers

If you search like this:
compare query with ALL 100,000 vectors
That is slow
So what do we do instead?

We organize the data first, so search becomes fast.

 That’s what Phase 1 does.
What Phase 1 is really doing (simple idea)

“Before searching, we arrange all vectors into groups so we don’t have to check everything.”

Real-life example

Think of a library:

Without organization:
All books in one pile
You check every book → slow
With organization:
Books split into sections (science, history, etc.)
You go to one section → fast

FAISS does the same thing with vectors

In [43]:
dimension = 1536
#Each item = 1536 numbers
n_vectors = 100_000
#You plan to store 100k items
n_clusters = 512
#Split all data into 512 groups
n_subvectors = 96
#split vector into 96 parts
bits_per_code = 8
#each part stored using 8 bits

quantizer = faiss.IndexFlatL2(dimension)
#decide which group a vector belongs to

index = faiss.IndexIVFFlat(quantizer, dimension, n_clusters)
#This creates a system that:
#divides vectors into clusters
#searches only a few clusters instead of all
print(index.is_trained)  # False

False


Why this phase is important

Without this step:

every search checks all vectors → slow

With this step:

search checks only a few clusters → fast
Key idea: Speed vs Accuracy
More clusters → faster and more accurate
Fewer clusters → slower and less precise

Phase 2 — Training the Index (Teaching It How to Organize Data)

At this point, we have created the index structure.

But there’s a problem:

The index exists, but it doesn’t know how to group the data yet.

That’s why index.is_trained was False.

What does “training” mean here?

Training does not mean machine learning like models.

Here, training simply means:

“Learn how to divide the data into clusters.”

Simple analogy

Think of Phase 1 like building shelves in a library.

Now in Phase 2:

We decide which books go on which shelf.
What happens if we skip training?

If you try to add data before training:

Error: Index not trained

👉 Because FAISS doesn’t know how to organize vectors yet.

When we first wrote the code, we used:

training_sample = np.random.rand(n_train, dimension).astype("float32")

This creates completely random vectors.

Why this was a problem

Random data has no real structure.

That means:

vectors are not meaningfully related
clusters become random
FAISS learns nothing useful

Result:

Very low recall (bad search quality)
What we changed

We replaced it with:

training_sample = all_vectors[:n_train]
Why this is better

Now:

training data comes from the actual dataset
FAISS sees real patterns
clusters become meaningful

Result:

Better grouping → better search → higher recall
Simple analogy
Before:

Teaching a librarian using random books with no categories.

After:

Teaching using actual books from the library.

Now the librarian can organize properly.

Important takeaway

“The index should be trained on data that looks like what it will later store and search.”

In [44]:
n_train = 50000
training_sample = all_vectors[:n_train]
#We don’t need all 100,000 vectors for training. A subset is enough to learn patterns.

index.train(training_sample)
#This is the most important line in this phase.

#What happens here:

#FAISS looks at the vectors
#finds natural groupings
#creates cluster centers

print(index.is_trained)  # True
#Now the index is ready to accept data.

True


Phase 3 — Adding Data to the Index (Storing Vectors + Keeping Them Linked)

Now that the index is trained, we can finally add our data.

👉 This is where the system actually becomes usable.
What are we doing in this phase?

“We store all vectors inside FAISS and make sure we can map them back to real data.”

⚠️ Important rule

FAISS:

stores vectors
assigns IDs automatically

But:

it does NOT store your product info

👉 So we must keep metadata in sync

In [45]:
all_vectors = np.random.rand(n_vectors, dimension).astype("float32")
#This creates:

#100,000 vectors
#each vector = 1536 numbers

batch_size = 10000

#Instead of adding everything at once, we add in chunks.

#Why batching?
#uses less memory
#works better for large datasets
#useful for streaming data

for start in range(0, n_vectors, batch_size):
  #This splits data into chunks like:
  #0–9999, 10000–19999, ...
    end = start + batch_size
    batch = all_vectors[start:end]
    #One chunk of vectors at a time

    first_id = index.ntotal
    #ntotal = number of vectors already stored
    #What ID the next vector will get
    index.add(batch)
    # FAISS:

     #stores vectors
     #assigns IDs automatically

#Example:

#IDs: 0 → 9999, then 10000 → 19999, etc.

    for i, fid in enumerate(range(first_id, index.ntotal)):
      #Loop through newly assigned IDs
        metadata_store[fid] = {
            "title": f"Product {fid}",
            "category": "audio" if fid % 2 == 0 else "peripherals",
        }
        #For each vector, we store:

        #ID → actual product info

print(index.ntotal)
print(len(metadata_store))

100000
100000


Why this step is critical

If you skip this:

👉 FAISS returns:

ID = 62139

But you won’t know:

What product is this?

Common bug

If these numbers are different:

FAISS data ≠ metadata

👉 Your system is broken.

💡 What this phase achieves

After this phase:

all vectors are stored
every vector has a matching metadata entry
system is ready for search
📚 Simple analogy

Think of this like:

FAISS = warehouse storing boxes
metadata_store = labels on boxes

👉 Without labels, boxes are useless.

Phase 4 — Searching (Finding Similar Items)

Now that:

the index is built ✅
trained ✅
and filled with data ✅

we can finally search.

What are we doing in this phase?

“Given a query vector, find the most similar vectors stored in FAISS.”

In [46]:
index.nprobe = 100
#How many clusters should FAISS search?
#Why this matters

#Remember:

#Data is split into clusters
#FAISS does NOT search all clusters

#So:

#Low nprobe → search fewer clusters → faster but less accurate
#High nprobe → search more clusters → slower but more accurate

query = np.random.rand(1, dimension).astype("float32")
#what the user is searching for
k = 5
#We want the top 5 most similar items

distances, ids = index.search(query, k)
#FAISS returns two things:

#ids → IDs of closest vectors
#distances → how close they are

for i, (fid, dist) in enumerate(zip(ids[0], distances[0]), 1):
    print(i, fid, dist, metadata_store.get(fid))
    #For each result:

#get the ID
#look it up in metadata_store
#print real product info

1 41563 226.02628 {'title': 'Product 41563', 'category': 'peripherals'}
2 21889 228.86597 {'title': 'Product 21889', 'category': 'peripherals'}
3 54144 229.28957 {'title': 'Product 54144', 'category': 'audio'}
4 68295 229.46976 {'title': 'Product 68295', 'category': 'peripherals'}
5 65498 229.67783 {'title': 'Product 65498', 'category': 'audio'}


62139 → closest match
222.15 → distance (lower = better)
metadata → actual product
mportant note

Distances may look large (like 200+).

👉 That’s normal because:

vectors are high-dimensional (1536 numbers)
we’re using random data
🔥 What this phase achieves

After this phase:

we can search for similar items
we get ranked results
we convert IDs into meaningful output

Phase 5 — Saving and Loading the Index (Don’t Lose Your Work)

At this point, you’ve built and filled your FAISS index.

But there’s a problem:

FAISS stores everything in RAM (memory).

👉 If your notebook stops or crashes, everything is lost.

The problem

If we do this:

faiss.write_index(index, INDEX_PATH)

and the process crashes midway:

👉 You may end up with:

A corrupted file (and no backup)
✅ The safe approach
2. Save to a temporary file first
faiss.write_index(index, TMP_PATH)

👉 Writes the full index safely to a temp file

In [47]:
INDEX_PATH = "index.faiss"
TMP_PATH = "index.tmp"
#INDEX_PATH → final saved file
#TMP_PATH → temporary file used during saving

faiss.write_index(index, TMP_PATH)
os.replace(TMP_PATH, INDEX_PATH)
#Writes the full index safely to a temp file
#This step is atomic (all-or-nothing):

#If it succeeds → new file replaces old one
#If it fails → old file remains safe

index_loaded = faiss.read_index(INDEX_PATH)
#Now the index is back in memory
index_loaded.nprobe = 20
#Important:

#nprobe is NOT saved in the file
#You must set it again after loading

print(index_loaded.ntotal)
#Confirms:

#all vectors are loaded
#nothing is lost

100000


Phase 6 — Measuring Recall (How Good Is Our Search?)

Now we have a working search system.

But a big question remains:

Are the results actually correct?

👉 This is where recall comes in.

🧠 What is recall?

Recall measures:

“How many of the correct results did our FAISS search find?”

📚 Simple example

Let’s say:

True top 10 results:
[A, B, C, D, E, F, G, H, I, J]
FAISS returns:
[A, B, X, Y, Z, ...]

👉 Common items = A, B

So:

recall = 2 / 10 = 0.2

In [48]:
exact_index = faiss.IndexFlatL2(dimension)
exact_index.add(all_vectors)
#This index:

#checks every vector
#gives 100% correct results
#but is slow

#👉 We use it as ground truth

In [49]:
n_eval = 200
k = 10

eval_queries = all_vectors[:n_eval]
#We use real vectors from the dataset
#👉 This makes evaluation meaningful

In [50]:
_, true_ids = exact_index.search(eval_queries, k)
#These are the correct answers

In [51]:
for nprobe in [5, 10, 20, 40, 64]:
    index.nprobe = nprobe
    _, approx_ids = index.search(eval_queries, k)
    #We test different values of nprobe

    recall_scores = []
    #Measures overlap between:

#true results
#FAISS results

    for q in range(n_eval):
        true_set = set(true_ids[q])
        approx_set = set(approx_ids[q])

        recall = len(true_set & approx_set) / k
        recall_scores.append(recall)

    avg_recall = np.mean(recall_scores)
    print(f"nprobe={nprobe} → recall@{k} = {avg_recall:.3f}")

nprobe=5 → recall@10 = 0.160
nprobe=10 → recall@10 = 0.216
nprobe=20 → recall@10 = 0.315
nprobe=40 → recall@10 = 0.465
nprobe=64 → recall@10 = 0.614


As nprobe increases:

recall improves (better accuracy)
search becomes slower